In [ ]:
import sys
from pathlib import Path

# Walk up to find root and anchor src/
current = Path.cwd().resolve()
REPO_ROOT = current.parent if current.name == "notebooks" else current
src_dir = REPO_ROOT / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import json
from PIL import Image
from ev_occupancy.env import CONFIG_SPOTS_FILE, CONFIGS_DIR, DATA_RAW_DIR

print(f"Project Root: {REPO_ROOT}")
print(f"Target Config: {CONFIG_SPOTS_FILE}")
print(f"Raw Images Dir: {DATA_RAW_DIR}")

In [ ]:
# Load the first image from data/raw_screenshots, or specify a sample path
valid_exts = {".png", ".jpg", ".jpeg"}
screenshots = sorted([p for p in DATA_RAW_DIR.iterdir() if p.suffix.lower() in valid_exts])

if screenshots:
    sample_path = screenshots[0]
    print(f"Found {len(screenshots)} screenshots. Using: {sample_path.name}")
    ref_img = Image.open(sample_path).convert("RGB")
else:
    # Fallback placeholder if screenshots haven't been copied into data/raw_screenshots yet
    print(f"No screenshots in {DATA_RAW_DIR}. Creating placeholder 1920x1080 canvas...")
    ref_img = Image.new("RGB", (1920, 1080), color=(30, 35, 45))

img_w, img_h = ref_img.size
print(f"Image Resolution: {img_w}x{img_h} px")

In [ ]:
DEFAULT_W, DEFAULT_H = 85, 115
spots_config = {}

if CONFIG_SPOTS_FILE.exists():
    print(f"Loading existing coordinates from: {CONFIG_SPOTS_FILE}")
    with open(CONFIG_SPOTS_FILE) as f:
        spots_config = json.load(f)
else:
    print("Generating default 2x10 grid seed...")
    spot_id = 1
    START_X, START_Y = 120, 200
    GAP_X, GAP_Y = 30, 45

    for row in range(2):
        for col in range(10):
            name = f"spot_{spot_id:02d}"
            is_hc = spot_id in [1, 2]  # Seeds spots 1 and 2 as handicapped
            spots_config[name] = {
                "x": int(START_X + col * (DEFAULT_W + GAP_X)),
                "y": int(START_Y + row * (DEFAULT_H + GAP_Y)),
                "w": DEFAULT_W + (25 if is_hc else 0),
                "h": DEFAULT_H,
                "type": "handicapped" if is_hc else "standard"
            }
            spot_id += 1

print(f"Tracking {len(spots_config)} stalls.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output

spot_selector = widgets.Dropdown(options=list(spots_config.keys()), value="spot_01", description="Active Spot:")
x_slider = widgets.IntSlider(min=0, max=img_w, step=1, description="X (Left):")
y_slider = widgets.IntSlider(min=0, max=img_h, step=1, description="Y (Top):")
w_slider = widgets.IntSlider(min=10, max=500, step=1, description="Width:")
h_slider = widgets.IntSlider(min=10, max=500, step=1, description="Height:")
type_toggle = widgets.ToggleButtons(options=["standard", "handicapped"], description="Type:")
save_btn = widgets.Button(description="Save Config to Disk", button_style="success", icon="save")
status_label = widgets.Label(value="")

def sync_sliders(spot_name):
    cfg = spots_config[spot_name]
    x_slider.value, y_slider.value = cfg["x"], cfg["y"]
    w_slider.value, h_slider.value = cfg["w"], cfg["h"]
    type_toggle.value = cfg.get("type", "standard")

sync_sliders(spot_selector.value)
plot_out = widgets.Output()

def render_canvas():
    with plot_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(14, 8), dpi=100)
        ax.imshow(ref_img)
        active = spot_selector.value

        for name, cfg in spots_config.items():
            is_active = (name == active)
            is_hc = (cfg.get("type") == "handicapped")
            edge_color = "#00FFFF" if is_active else ("#FFA500" if is_hc else "#00FF66")
            lw = 2.5 if is_active else 1.2

            rect = patches.Rectangle((cfg["x"], cfg["y"]), cfg["w"], cfg["h"],
                                     linewidth=lw, edgecolor=edge_color, facecolor="none")
            ax.add_patch(rect)
            ax.text(cfg["x"], max(0, cfg["y"] - 4), f"{name} ({'HC' if is_hc else 'STD'})",
                    color="black", fontsize=8, fontweight="bold",
                    bbox=dict(facecolor=edge_color, edgecolor="none", pad=1.5, alpha=0.85))

        ax.set_title(f"Calibrating: {active} | Type: {spots_config[active]['type'].upper()}", fontsize=11, fontweight="bold")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

def on_spot_change(change):
    sync_sliders(change["new"])
    render_canvas()

def on_param_change(change):
    active = spot_selector.value
    spots_config[active]["x"] = x_slider.value
    spots_config[active]["y"] = y_slider.value
    spots_config[active]["w"] = w_slider.value
    spots_config[active]["h"] = h_slider.value
    spots_config[active]["type"] = type_toggle.value
    render_canvas()

def on_save(b):
    with open(CONFIG_SPOTS_FILE, "w") as f:
        json.dump(spots_config, f, indent=2)
    status_label.value = f"Saved {len(spots_config)} stalls to {CONFIG_SPOTS_FILE.name} ({CONFIG_SPOTS_FILE.stat().st_size} bytes)"

spot_selector.observe(on_spot_change, names="value")
x_slider.observe(on_param_change, names="value")
y_slider.observe(on_param_change, names="value")
w_slider.observe(on_param_change, names="value")
h_slider.observe(on_param_change, names="value")
type_toggle.observe(on_param_change, names="value")
save_btn.on_click(on_save)

display(widgets.VBox([
    widgets.HBox([spot_selector, type_toggle]),
    widgets.HBox([x_slider, y_slider]),
    widgets.HBox([w_slider, h_slider]),
    widgets.HBox([save_btn, status_label])
]), plot_out)

render_canvas()

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(14, 10), dpi=100)
axes = axes.flatten()

for idx, (name, cfg) in enumerate(spots_config.items()):
    crop = ref_img.crop((cfg["x"], cfg["y"], cfg["x"] + cfg["w"], cfg["y"] + cfg["h"]))
    ax = axes[idx]
    ax.imshow(crop)
    tag = " [HC]" if cfg.get("type") == "handicapped" else ""
    ax.set_title(f"{name}{tag}", fontsize=9, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sanity Check: 20 Isolated Crops", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()